### Docker
Docker = система управления контейнерами

Контейнер = процесс, изолированный по ресурсам и с преднастроенным окружением<br>
Образ (Image) = снепшот файловой системы

Допустим есть некое приложение app, которое нужно запустить `sh app`. Для универсализации процесса запуска его можно обернуть в контейнер и запускать уже обертку, которая под капотом отвечает за настройку ресурсов `docker run app`

Почему полезна контейнеризвация:
- Изоляция окружения<br>нет конфликта версий библиотек
- Переносимость (works everywhere)<br>одинаково ставится на разные версии Linux
- Быстрое масштабирование<br>дополнительный экземпляр процесса запускается за секунды
- Изоляция ресурсов<br>контролирование использование
- Повторяемость (reproducibility)<br>экономия на настройке
- Упрощённая доставка (CI/CD)<br>экономия на настройке при развертывании
- Безопасность<br>ограничиваются права, системных вызорвов
- Изоляция сети<br>снаружи видны как отдельные машины

Когда нужно запускать много экземпляров приложения и распределенно, возникает необхоимость делать обертку над оберткой - это уже Kubernetes. Логика та же, но уже в масштабе

Какие конкретно ресурсы ОС изолируются в рамках контейнера?
- с помощью утилиты __namespaces__ выделяется: свое дерево процессов PID, файловая система FS, доменное имя, сетевой менеджемент (iptables etc)
- с помощью утилиты __cgroups__ выделяются свои лимиты на ресурсы (cpu, ram, i/o, network)
- с помощью __OverlayFS__ создается layered файловая системы
- используются инструменты (например, sescomp), чтобы дропунть потенциально опасный функционал

*OverlayedFS - "слоеная" файловая система, когда есть read-only базовый слой, а все модификации наслаиваются в новых слоях. В базовом слое либо голая операционная система (точнее пользовательская ее часть User-space Linux, ядро)

Ядро Linux = драйверы, системные вызовы. Пользоватлеьская часть = shell, более выскоуровневые библиотеки

Можно запускать не только фоне, но и интерактивный процесс (флаг -i -t создает). Можно подключаться к работающему конейтнеру

Что происходит при docker run:
- идет обращение к оркестрирующему процессу __dockerd__, который создает в своей таблице запись про новый контейнер
- форкается новый процесc командой clone (сразу с изоляцией - с созданием новых пространств имен)
- собирается новая файловая система (mount overlay) и её каталог делается корневым (chroot)
- настраиваются лимиты (с помощью namespaces, cgroups, seccomp)
- запуск целевой команды через execve 

### Docker-compose
Docker-compose - более высокоуровневая обертка над Docker. Аналог Kubernetes. Он работает не с отдельными Docker-файлами, а со сложными конфигурациями сервисов, описываемыми в файле конфигурации (yml). Например FastAPI + Triton

В конфигурации указываются ключевые поля: services, image, volumes, ports, 

CI/CD регламентирует релиз новых версий приложения
Традиционно функционал CI/CD есть в GitLab
по событию (push, manual run) => сборка + тест + deploy на сервер(а)
Ansible: скрипт для менеджемнта парка машин
Kubernetes: для оркестрации

## Docker command

#### Запуск
```
# запустить контейнер
docker run <image>

# дать свое имя
docker run --name

# сделать port forwarding
docker run --p <host_port>:<image_port> <image>

# запустить как background-процесс
docker run --d

# 
docker run --hostname <host_name> <image>

docker run --it
```

#### Мониторинг
```
# вывести все запущенные на хосте контейнеры
docker ps

# вывести все запущенные контейнером процессы
docker top

# event-log по конетрнеру
docker logs

# накопленная статистика по запущенным контейнера
docker stats
```

#### Управление контейнерами
```
# выгрузить контейнер
docker rm <container>

# запустить контейнер заново
docker start <container>

# поставить выполнение на паузу
docker stop <container>

# выполнить кастомную команду внутри контейнера
docker exec --it <container> <bash commands>

# создать образ
docker commit <container>
```

#### Управление образов
```
# выгрузить образ в репозиторий 
docker push

# загрузить образ из репозитория
docker pull

# собрать образ из файла (запустить и завершить)
docker build

# вывести все зарегистрированные образы
docker images

# созхранить образ в файл
docker save

# загрузить образ из файла
docker load

```

## Repositories
Докер

## Dockerfile

Dockerfile - это текстовый набор инструкций для развертывания контейнера. В нем указываем какой образ файловой системы подключить

FROM
RUN
CMD
ADD
WORK

| Инструкция  | Назначение | Пример |
|-------------|------------|--------|
| **FROM**    | Указываем, какой image подключать | `FROM python:3.10` |
| **RUN**     | Кастомные команды ОС для установки пакетов / подготовки окружения | `RUN apt-get update && apt-get install -y git` |
| **COPY**    | Копирует файлы из локальной машины внутрь образа | `COPY app.py /app/app.py` |
| **ADD**     | Как COPY, но умеет распаковывать архивы и скачивать URL | `ADD archive.tar.gz /app/` |
| **WORKDIR** | Устанавливает рабочую директорию для последующих команд чтобы не писать абсолютные пути | `WORKDIR /app` |
| **CMD**     | Стартовая команда (что-то типа void main) | `CMD ["python", "app.py"]` |
| **ENTRYPOINT** | Базовая команда, которую сложно перезаписать; сочетается с CMD | Для обязательной основной логики | `ENTRYPOINT ["python"]` |
| **EXPOSE**  | Указывает, какой порт будет слушать приложение | `EXPOSE 8000` |
| **ENV**     | Устанавливает переменные окружения | `ENV APP_ENV=prod` |
| **VOLUME**  | Какой внешний каталог примонтировать для доступа из образа | `VOLUME /data` |
| **USER**    | Выполняет команды от имени указанного пользователя (для безопасности и изоляции) | `USER appuser` |
| **LABEL**   | Добавляет кастомные метаданные | `LABEL maintainer="you@example.com"` |


Возможно есть некоторая аналогия с CD-дисководом, мы вставляем нужный диск и ОС запускает его